# 2.1 RoPE 算子开发

## 前置要求

具备 C++、Ascend C、张量布局和三角函数基础；SIMD 主线使用 Ascend 910B3/CANN 9.0。

## 章节目标

- 解释 interleaved `[12,64]` 与 pair-planar even/odd `[384]` 的转换；
- 运行 Ascend C SIMD/Vector Core RTC 链路；
- 对照 Ascend 950 SIMT 模板理解线程级映射，并遵守硬件证据边界。

RoPE 将相邻两个通道组成旋转对：

\[
y_{even}=x_{even}\cos\theta-x_{odd}\sin\theta,\quad
y_{odd}=x_{even}\sin\theta+x_{odd}\cos\theta
\]

每个 head 有 64 个通道，即 32 个旋转 pair；12 个 head 共 384 个 pair。Host 把 interleaved 输入拆成连续 even/odd 数组，Kernel 计算后再重排回原布局。

<img src="images/rope_pair_planar_flow.svg" width="760" style="display:block; margin-left:0;" />


## 两条编程路径

<table style="text-align:left; margin-left:0;">
<tr><th>路径</th><th>并行单位</th><th>输入/输出合同</th><th>本实验状态</th></tr>
<tr><td>SIMD</td><td>Vector Core 对 LocalTensor 执行向量指令</td><td>6 个连续 <code>float[384]</code> 缓冲区</td><td>910B3 正式主线：AIV + RTC</td></tr>
<tr><td>SIMT</td><td>一个线程处理一个旋转 pair</td><td>同一 pair-planar 语义，额外传入 count</td><td>仅提供 950 模板，状态为 DEFERRED</td></tr>
</table>

<img src="images/rope_simd_simt_boundary.svg" width="760" style="display:block; margin-left:0;" />

> **证据边界：**910B3 只能验收 SIMD/AIV + RTC；`rope_simt_950.asc` 必须在 Ascend 950 上实际编译、执行并校验，不能用编译器 help、模板文件存在或 910B3 SIMD 结果代替。


## 章节内容

<table style="text-align:left; margin-left:0;">
<tr><th>小节</th><th>内容</th><th>入口</th></tr>
<tr><td>2.2</td><td>精选 SIMD/SIMT 代码、RTC 编译运行和结果字段</td><td><a href="./02.02_rope_simd_simt.ipynb">02.02_rope_simd_simt.ipynb</a></td></tr>
<tr><td>2.3</td><td>客观题与简单/中等/困难三档实践</td><td><a href="./02.03_chapter_test.ipynb">02.03_chapter_test.ipynb</a></td></tr>
</table>
